# An-Ra V4 — Protected Kaggle P100 Trainer

This notebook is a sequential checkpoint-baton worker for the canonical 181M-parameter V4 foundation. It restores one verified full-resume checkpoint from a **private Kaggle Dataset**, automatically selects the data window containing the checkpoint's next token, trains on one P100, and publishes exactly one replacement checkpoint under `/kaggle/working/ANRA_KAGGLE_EXPORT`.

Before **Run All**: select **Accelerator → GPU P100**, enable Internet for the GitHub checkout, attach one private Dataset snapshot containing `anra-v4-current-full-resume.pt`, its JSON metadata, and the required continuation archive. Never run this notebook while a Colab canonical trainer is active. Kaggle input is read-only, so after completion save/download the output and replace the shared Drive checkpoint before the next worker starts.

In [ ]:
# Operator configuration
WORKER_ID = 'kaggle-p100-primary'
REPO_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
REPO_REF = 'iterate500'
SESSION_BUDGET_MINUTES = 480
DRAIN_RESERVE_MINUTES = 30
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8
PACK_CATALOG = [
    {
        'name': 'v4_phase_a_170m_seed1301', 'start_token': 0, 'end_token': 170_000_000,
        'archive_sha256': '07f01bf4809667acc670eb9c94dfab38d28522d7bfc2d4c930e71898cff86ee7',
        'files': [
            {'name': 'v4_phase_a_170m_seed1301.tar.gz.part00', 'size': 83886080, 'sha256': '9efe814598f52275dee15cb70e981e1bb375e24dbf97f39788aa4c84498f33f0'},
            {'name': 'v4_phase_a_170m_seed1301.tar.gz.part01', 'size': 63233323, 'sha256': 'c073e325d2fbe09fe4afefe75c251db59540ea3358e34d8a184d7bb2831e0f6a'},
        ],
    },
    {
        'name': 'v4_phase_a_cont_170m_to_500m_seed1301', 'start_token': 170_000_000, 'end_token': 500_000_000,
        'archive_sha256': '330ac75a2dbe20a3bd6608adf1c7325fd4a8bf7e81be546ccd8e5f93877f4888',
        'files': [
            {'name': 'v4_phase_a_cont_170m_to_500m_seed1301.tar.gz', 'size': 276647985, 'sha256': '330ac75a2dbe20a3bd6608adf1c7325fd4a8bf7e81be546ccd8e5f93877f4888'},
        ],
    },
]
print({'worker': WORKER_ID, 'training_minutes': SESSION_BUDGET_MINUTES - DRAIN_RESERVE_MINUTES})

In [ ]:
# Enforce the requested Kaggle accelerator.
import os, pathlib, shutil, subprocess, time, json, gc, torch
assert torch.cuda.is_available(), 'No GPU. In Notebook options select Accelerator: GPU P100.'
gpu_name = torch.cuda.get_device_name(0)
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert 'P100' in gpu_name.upper(), f'Expected a P100, received {gpu_name}. Change the Kaggle accelerator.'
assert total_gib >= 14, f'P100 memory contract failed: {total_gib:.1f} GiB'
subprocess.run(['nvidia-smi'], check=True)
print(f'READY: {gpu_name}, {total_gib:.1f} GiB')

In [ ]:
# Clone one clean operational checkout.
REPO = pathlib.Path('/kaggle/working/anra')
os.chdir('/kaggle/working')
git_env = os.environ.copy(); git_env['GIT_TERMINAL_PROMPT'] = '0'
clone = ['git', '-c', 'http.version=HTTP/1.1', 'clone', '--depth', '1', '--single-branch', '--branch', REPO_REF, REPO_URL, str(REPO)]
for attempt in range(1, 4):
    if REPO.exists(): shutil.rmtree(REPO)
    result = subprocess.run(clone, text=True, capture_output=True, env=git_env)
    if result.returncode == 0: break
    print(f'Clone attempt {attempt}/3 failed: {result.stderr.strip()}')
    if attempt == 3: raise RuntimeError('Unable to clone An-Ra. Confirm Kaggle Internet is enabled.')
    time.sleep(2 ** attempt)
os.chdir(REPO)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert not subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip()
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print(f'Clean source: {REPO_REF}@{commit}')

In [ ]:
# Resolve exactly one verified private Dataset snapshot and copy its checkpoint to local scratch.
from runtime.safe_load import safe_torch_load
from training.kaggle_assets import resolve_kaggle_training_assets, sha256_file
ASSETS = resolve_kaggle_training_assets('/kaggle/input')
SCRATCH = pathlib.Path('/kaggle/working/anra-scratch'); SCRATCH.mkdir(parents=True, exist_ok=True)
EXPORT_ROOT = pathlib.Path('/kaggle/working/ANRA_KAGGLE_EXPORT'); EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
resume_checkpoint = SCRATCH / 'resume-source.pt'
temporary = resume_checkpoint.with_suffix('.pt.tmp')
shutil.copyfile(ASSETS.checkpoint, temporary)
assert temporary.stat().st_size == ASSETS.checkpoint.stat().st_size
assert sha256_file(temporary) == ASSETS.checkpoint_sha256
temporary.replace(resume_checkpoint)
resume_payload = safe_torch_load(resume_checkpoint, map_location='cpu')
assert isinstance(resume_payload, dict), 'Full-resume checkpoint is not a mapping'
resume_step = int(resume_payload.get('global_step', resume_payload.get('step', -1)))
counts = resume_payload.get('continuation_token_counts', {})
phase_a_tokens_seen = int(counts.get('A', resume_payload.get('tokens_processed', 0)))
assert resume_step >= 0 and phase_a_tokens_seen >= 0
del resume_payload; gc.collect()
print(f'Verified foundation checkpoint: step={resume_step:,} phase_A_tokens={phase_a_tokens_seen:,} sha256={ASSETS.checkpoint_sha256}')

In [ ]:
# Load signing identity from Kaggle Secrets, with a private-Dataset compatibility fallback.
manifest_key = evidence_key = ''
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    manifest_key = secrets.get_secret('ANRA_MANIFEST_SIGNING_KEY') or ''
    evidence_key = secrets.get_secret('ANRA_EVIDENCE_SIGNING_KEY') or ''
except Exception:
    pass
if (len(manifest_key) < 64 or len(evidence_key) < 64) and ASSETS.signing_key is not None:
    private_keys = json.loads(ASSETS.signing_key.read_text(encoding='utf-8-sig'))
    manifest_key = str(private_keys.get('manifest', ''))
    evidence_key = str(private_keys.get('evidence', ''))
assert len(manifest_key) >= 64 and len(evidence_key) >= 64, 'Add both signing keys as Kaggle Secrets or to the private input snapshot.'
os.environ['ANRA_MANIFEST_SIGNING_KEY'] = manifest_key
os.environ['ANRA_EVIDENCE_SIGNING_KEY'] = evidence_key
os.environ['ANRA_REQUIRE_SIGNED_EVIDENCE'] = '1'
print('Signing identity loaded without disclosure.')

In [ ]:
# Materialize only the required immutable token window and sign this Kaggle launch.
from training.colab_continuation import materialize_continuation_pack, select_continuation_pack
selected_pack = select_continuation_pack(phase_a_tokens_seen, PACK_CATALOG)
PACK_ROOT = materialize_continuation_pack(training_home=ASSETS.training_home, scratch_root=SCRATCH, pack_parent=REPO / 'output' / 'v2' / 'cloud_packs', pack=selected_pack)
launch = REPO / 'output' / 'v2' / 'launch_manifests' / f'{WORKER_ID}.json'
artifact = SCRATCH / f'anra-v4-{WORKER_ID}.pt'
subprocess.run([
    'python', '-m', 'scripts.create_cloud_launch', '--pack-root', str(PACK_ROOT),
    '--output', str(launch), '--artifact-path', str(artifact),
    '--checkpoint-source', str(resume_checkpoint), '--worker-id', WORKER_ID,
    '--runtime-estimate-hours', str(SESSION_BUDGET_MINUTES / 60),
    '--batch-size', str(BATCH_SIZE), '--accumulation', str(GRADIENT_ACCUMULATION),
], check=True)
signed = json.loads(launch.read_text(encoding='utf-8'))
assert signed['git_commit'] == commit
print({'pack': selected_pack.name, 'window': signed['token_window'], 'checkpoint': signed['checkpoint_source_hash']})

In [ ]:
# Run the only canonical writer and publish one portable Kaggle output checkpoint.
os.environ['ANRA_SHARED_CHECKPOINT_DIR'] = str(EXPORT_ROOT)
os.environ['ANRA_DURABILITY_OUTBOX'] = str(SCRATCH / 'durability-outbox')
os.environ['ANRA_DURABILITY_REPLICAS'] = json.dumps([
    {'name': 'kaggle-output', 'path': str(EXPORT_ROOT), 'kind': 'single_file', 'canonical': True}
])
os.environ['ANRA_DURABILITY_MIN_PROTECTED_REPLICAS'] = '1'
os.environ['ANRA_DURABILITY_COPY_STREAMS'] = '2'
os.environ['ANRA_DURABILITY_ACK_TIMEOUT_SECONDS'] = '1800'
os.environ['ANRA_CHECKPOINT_EVERY_MIN'] = '60'
os.environ['ANRA_DURABLE_CHECKPOINT_STEPS'] = '200'
train_command = [
    'python', '-u', '-m', 'training.train_unified', '--mode', 'session',
    '--launch-manifest', str(launch), '--prepare_data', 'never',
    '--post-session-eval', 'none', '--data_path', 'training_data/anra_training.txt',
]
process = subprocess.Popen(train_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
assert process.stdout is not None
from collections import deque
tail = deque(maxlen=160)
for line in process.stdout:
    tail.append(line); print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    failure = ''.join(tail)
    (EXPORT_ROOT / 'latest_training_failure.log').write_text(failure, encoding='utf-8')
    print('\n[KAGGLE TRAINER FAILURE — LAST 160 LINES]\n' + failure)
    raise RuntimeError(f'Kaggle trainer exited with status {return_code}')
result_checkpoint = EXPORT_ROOT / 'anra-v4-current-full-resume.pt'
result_metadata = EXPORT_ROOT / 'anra-v4-current-full-resume.json'
assert result_checkpoint.is_file() and result_metadata.is_file()
pointer = json.loads(result_metadata.read_text(encoding='utf-8'))
assert result_checkpoint.stat().st_size == int(pointer['size_bytes'])
assert sha256_file(result_checkpoint) == str(pointer['sha256'])
print('KAGGLE SESSION COMPLETE. Save this notebook version or download ANRA_KAGGLE_EXPORT, then replace the shared Drive checkpoint pair before the next trainer starts.')